# ce-train
Generated from `code/business_entity_resolution/src/cross_encoder_train.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0 lightgbm==4.7.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/cross_encoder_score.py
"""Cross-encoder re-scoring of borderline pairs (experiment; branch cross-encoder).

Model: distilbert-base-multilingual-cased (Apache-2.0), fine-tuned by cross_encoder_train.py as a binary pair
classifier. Input: "{business_name} | {business_address} | {country}" of the S1 record [SEP] the same for the
candidate, raw organiser text, max 128 tokens.
Blend (borderline pairs only, BAND[0] < p_model < BAND[1]): p = (1 - w) * p_model + w * p_ce; other pairs keep
p_model. Then the usual selection (threshold, margin, one owner per S2/S3).

Test usage (GPU):
  python src/cross_encoder_score.py --input DIR --ce-dir DIR --scores test_scores.parquet --config ce_config.json \
      --out-dir output/ce_blend [--validator utils/validate_submission.py] [--current output/matching_results.tsv]
"""
from __future__ import annotations

import argparse
import json
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:
    sys.path.insert(0, "/tmp/src")

MODEL = "distilbert-base-multilingual-cased"
MAX_LEN = 128
BAND = (0.02, 0.98)
T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - T0:6.0f}s] {msg}", flush=True)


def find(root: Path, name: str, not_parent: tuple = ("normalised", "features_extra", "aug")) -> Path:
    hits = sorted(p for p in root.rglob(name) if p.parent.name not in not_parent)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


def texts(root: Path, side: str, ids: pl.Series) -> dict[str, str]:
    """entity_id -> "name | address | country" from the organiser files of one side (train / test)."""
    out = {}
    for k in (1, 2, 3):
        d = (pl.scan_parquet(find(root, f"{side}_source{k}.parquet")).select("entity_id", "business_name", "business_address", "country")
             .filter(pl.col("entity_id").is_in(ids.implode())).collect())
        t = d.select("entity_id", pl.concat_str([pl.col("business_name").fill_null(""), pl.col("business_address").fill_null(""),
                                                 pl.col("country").fill_null("")], separator=" | ").alias("t"))
        out.update(zip(t["entity_id"].to_list(), t["t"].to_list()))
    return out


def score_pairs(model, tok, pairs: pl.DataFrame, txt: dict, batch: int = 512, log_every: int = 200) -> np.ndarray:
    import torch
    model.eval()
    a, b = [txt.get(s, "") for s in pairs["s1"].to_list()], [txt.get(c, "") for c in pairs["cand"].to_list()]
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(a), batch):
            enc = tok(a[i:i + batch], b[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
            out.append(torch.sigmoid(model(**enc).logits.float().squeeze(-1)).cpu().numpy())
            if log_every and (i // batch) % log_every == 0:
                log(f"  scored {i + len(a[i:i + batch]):,} / {len(a):,}")
    return np.concatenate(out) if out else np.empty(0)


def blend(p_model: np.ndarray, p_ce: np.ndarray, w: float) -> np.ndarray:
    """p_ce: NaN outside the borderline band."""
    border = ~np.isnan(p_ce)
    p = p_model.copy()
    p[border] = (1 - w) * p_model[border] + w * p_ce[border]
    return p


def main() -> None:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from selection import select
    from submission import write_submission
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--ce-dir", required=True, type=Path, help="folder holding ce_model/ and ce_config.json (searched)")
    ap.add_argument("--scores", required=True, type=Path, help="test pair probabilities of the final model (searched if a folder)")
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--validator", default=None)
    ap.add_argument("--current", default=None, type=Path, help="current final matching_results.tsv to compare with")
    a = ap.parse_args()
    a.out_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = next(a.ce_dir.rglob("ce_config.json"))
    cfg = json.loads(cfg_path.read_text())
    mdir = cfg_path.parent / "ce_model"
    sp = a.scores if a.scores.is_file() else next(a.scores.rglob("test_scores.parquet"))
    st = pl.read_parquet(sp)
    border = st.filter((pl.col("p") > BAND[0]) & (pl.col("p") < BAND[1]))
    log(f"test pairs {st.height:,}; borderline {border.height:,} ({border.height / st.height:.1%}); w = {cfg['w']}, selection {cfg['selection']}")
    tok = AutoTokenizer.from_pretrained(mdir)
    model = AutoModelForSequenceClassification.from_pretrained(mdir).to("cuda")
    txt = texts(a.input, "test", pl.concat([border["s1"], border["cand"]]).unique())
    t0 = time.time()
    p_ce = score_pairs(model, tok, border.select("s1", "cand"), txt)
    log(f"scored {border.height:,} borderline test pairs in {time.time() - t0:.0f}s ({border.height / max(time.time() - t0, 1e-9):.0f} pairs/s)")
    st = st.join(border.select("s1", "cand").with_columns(pl.Series("p_ce", p_ce)), on=["s1", "cand"], how="left")
    st = st.with_columns(pl.Series("p_blend", blend(st["p"].to_numpy(), st["p_ce"].fill_null(np.nan).to_numpy(), cfg["w"])))
    st.write_parquet(a.out_dir / "test_scores_blend.parquet")
    matched = select(st.select("s1", "cand", pl.col("p_blend").alias("p")), **{**cfg["selection"], "one_owner": True})
    s1_all = pl.read_parquet(find(a.input, "test_source1.parquet"), columns=["entity_id", "country"]).rename({"entity_id": "s1"})
    cand = pl.read_parquet(find(a.input, "test_candidates.parquet"), columns=["s1", "cand", "p_u50"])
    lists = cand.sort(["s1", "p_u50", "cand"], descending=[False, True, False], nulls_last=True).group_by("s1", maintain_order=True).agg(pl.col("cand"))
    ml = matched.sort(["s1", "p"], descending=[False, True]).group_by("s1", maintain_order=True).agg(pl.col("cand"))
    write_submission(a.out_dir, s1_all["s1"].to_list(), dict(zip(ml["s1"].to_list(), ml["cand"].to_list())),
                     dict(zip(lists["s1"].to_list(), lists["cand"].to_list())))
    rep = [f"# Cross-encoder blend on test (w = {cfg['w']})\n", f"Borderline pairs scored: {border.height:,}. Matches: {matched.height:,}.\n"]
    if a.validator:
        import importlib.util
        spec = importlib.util.spec_from_file_location("vs", a.validator); vs = importlib.util.module_from_spec(spec); spec.loader.exec_module(vs)
        with tempfile.TemporaryDirectory() as td:
            for k in (1, 2, 3):
                pl.read_parquet(find(a.input, f"test_source{k}.parquet"), columns=["entity_id"]).write_csv(Path(td) / f"test_source{k}.tsv", separator="\t")
            e, w = vs.validate(str(a.out_dir / "matching_results.tsv"), str(a.out_dir / "candidate_pairs.tsv"), td, check_ids=True)
        rep.append(f"Validator (--check-ids): {len(e)} errors, {len(w)} warnings {e[:3]}\n")
        log(f"validator: {len(e)} errors, {len(w)} warnings")
    n = s1_all.join(matched.group_by("s1").len("n"), on="s1", how="left").with_columns(pl.col("n").fill_null(0))
    pc = n.group_by("country").agg(pl.col("n").mean().alias("matches_per_s1"), (pl.col("n") == 0).mean().alias("share_no_match")).sort("country")
    rep += ["## Matches per S1\n", str(pc) + "\n"]
    new = matched.select("s1", "cand").with_columns(pl.lit(True).alias("m"))
    fr = s1_all.filter(pl.col("country") == "France").sample(10, seed=42)
    ex = fr.join(matched, on="s1", how="left").join(st.select("s1", "cand", "p", "p_ce"), on=["s1", "cand"], how="left")
    t_fr = texts(a.input, "test", pl.concat([ex["s1"], ex["cand"].drop_nulls()]).unique())
    ex = ex.with_columns(pl.col("s1").replace_strict(t_fr, default="").alias("s1_text"), pl.col("cand").replace_strict(t_fr, default="").alias("cand_text"))
    rep += ["## 10 random French S1\n", str(ex.select("s1_text", "cand_text", "p", "p_ce")) + "\n"]
    if a.current and a.current.exists():
        cur = pl.read_csv(a.current, separator="\t", quote_char=None, infer_schema=False).select(
            pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").fill_null("").str.split(",").list.eval(pl.element().filter(pl.element() != "")).list.sort().alias("m0"))
        nm = s1_all.join(matched.group_by("s1").agg(pl.col("cand").sort().alias("m1")), on="s1", how="left").with_columns(pl.col("m1").fill_null([]))
        ch = nm.join(cur, on="s1").with_columns((pl.col("m1") != pl.col("m0")).alias("changed")).group_by("country").agg(
            pl.col("changed").sum().alias("s1_changed"), pl.col("changed").mean().alias("share")).sort("country")
        rep += ["## Change vs the current final\n", str(ch) + "\n"]
    (a.out_dir / "ce_test_report.md").write_text("\n".join(rep))
    log("done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/src/selection.py
"""Match selection: from scored candidate pairs to the predicted matches per S1, tuned for macro F0.5.

Rules (all tuned on validation, never on test):
- threshold: keep a pair only if its match probability p >= tau;
- one owner: every S2/S3 record matches at most one S1 (EDA: 0 of 7.6M train records match two), so each
  candidate is kept only for the S1 that gives it the highest p;
- margin: keep a pair only if p >= margin * (best p of that S1), which drops weak extra matches next to a strong one.
F0.5 weighs precision twice, so a wrong match costs more than a missed one: the tuned threshold is usually high.

f05_table() is a vectorised version of metric.py (same definition, checked against it in the tests and in
run_model.py): macro F0.5 over ALL evaluated S1, an S1 without predictions scores 1.0 if it has no true match.
"""
from __future__ import annotations

import itertools

import polars as pl

TAUS = [round(0.05 * i, 2) for i in range(2, 20)]  # 0.10 .. 0.95
MARGINS = [0.0, 0.3, 0.5, 0.7]
BETA2 = 0.25


def select(scored: pl.DataFrame, tau: float, margin: float = 0.0, one_owner: bool = True, tau2: float | None = None,
           sim: float = 0.9, owner_delta: float = 0.0) -> pl.DataFrame:
    """scored: s1, cand, p (+ s_sim_name, s_num_eq for the secondary rule). Returns the kept (s1, cand, p) rows.
    tau2: also keep a pair with tau2 <= p < tau if it strongly resembles a confident candidate of the same S1
    (s_sim_name >= sim and the same first house number). owner_delta: with one_owner, an S2/S3 claimed by two S1
    goes to the higher p only if it leads by more than owner_delta; otherwise it is dropped from both."""
    keep = pl.col("p") >= tau
    if tau2 is not None and tau2 < tau:
        keep = keep | ((pl.col("p") >= tau2) & (pl.col("s_sim_name") >= sim) & (pl.col("s_num_eq") >= 1)).fill_null(False)
    x = scored.filter(keep)
    if one_owner:  # ties: the smaller s1 id wins (deterministic)
        x = x.sort(["cand", "p", "s1"], descending=[False, True, False])
        if owner_delta > 0:
            lead = pl.col("p") - pl.col("p").shift(-1).over("cand")
            x = x.with_columns(pl.int_range(pl.len()).over("cand").alias("_r"), lead.fill_null(1.0).alias("_lead"))
            x = x.filter((pl.col("_r") == 0) & (pl.col("_lead") > owner_delta)).drop("_r", "_lead")
        else:
            x = x.unique("cand", keep="first", maintain_order=True)
    if margin > 0:
        x = x.filter(pl.col("p") >= margin * pl.col("p").max().over("s1"))
    return x.select("s1", "cand", "p")


def f05_table(pred: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame) -> pl.DataFrame:
    """Per evaluated S1: n_pred, n_true, tp, precision, recall, f05. pred / truth: s1, cand; s1s: s1 (+ any
    attribute columns, e.g. country), the S1 to evaluate. S1 missing from pred count as empty predictions."""
    keys = s1s.select("s1")
    pred = pred.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    truth = truth.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    tp = pred.join(truth, on=["s1", "cand"], how="semi").group_by("s1").len("tp")
    x = (s1s.join(pred.group_by("s1").len("n_pred"), on="s1", how="left")
         .join(truth.group_by("s1").len("n_true"), on="s1", how="left")
         .join(tp, on="s1", how="left")
         .with_columns(pl.col("n_pred", "n_true", "tp").fill_null(0)))
    prec = pl.when(pl.col("n_pred") > 0).then(pl.col("tp") / pl.col("n_pred"))
    rec = pl.when(pl.col("n_true") > 0).then(pl.col("tp") / pl.col("n_true"))
    f = (pl.when(pl.col("n_true") == 0).then((pl.col("n_pred") == 0).cast(pl.Float64))
         .when(pl.col("tp") == 0).then(0.0)
         .otherwise((1 + BETA2) * prec * rec / (BETA2 * prec + rec)))
    return x.with_columns(prec.alias("precision"), rec.alias("recall"), f.alias("f05"))


def summary(tab: pl.DataFrame, by: str | None = None) -> pl.DataFrame:
    """Macro F0.5, mean precision / recall (over S1 where defined), per group and overall (+ singletons)."""
    aggs = [pl.len().alias("s1"), pl.col("f05").mean().alias("macro_f05"), pl.col("precision").mean().alias("precision"),
            pl.col("recall").mean().alias("recall"), pl.col("n_pred").mean().alias("pred_per_s1")]
    kind = pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton"))
    parts = [tab.select(pl.lit("overall").alias("group"), *aggs),
             tab.with_columns(kind.alias("_k")).group_by("_k").agg(aggs).sort("_k")
             .select(pl.format("kind={}", "_k").alias("group"), *[a.meta.output_name() for a in aggs])]
    if by:
        parts.insert(1, tab.group_by(by).agg(aggs).sort(by).select(pl.format(f"{by}={{}}", by).alias("group"),
                                                                     *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


def tune(scored: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame, taus=TAUS, margins=MARGINS,
         owners=(True, False)) -> pl.DataFrame:
    """Macro F0.5 on s1s for every (tau, margin, one_owner). Sorted best first."""
    rows = []
    for tau, margin, owner in itertools.product(taus, margins, owners):
        tab = f05_table(select(scored, tau, margin, owner), truth, s1s)
        rows.append({"tau": tau, "margin": margin, "one_owner": owner, "macro_f05": tab["f05"].mean(),
                     "pred_per_s1": tab["n_pred"].mean()})
    return pl.DataFrame(rows).sort("macro_f05", descending=True)


In [ ]:
%%writefile /tmp/src/submission.py
"""Write matching_results.tsv / candidate_pairs.tsv in the exact challenge format.

One row per S1 (in the given order), tab-separated, comma-joined IDs, no quoting,
no duplicates, empty second column when there is nothing to list.
"""
from collections.abc import Iterable, Mapping
from pathlib import Path

MATCHING_HEADER = ("source1_entity_id", "matched_entity_ids")
CANDIDATE_HEADER = ("source1_entity_id", "candidate_entity_ids")


def write_id_lists(path: Path, header: tuple[str, str], s1_ids: Iterable[str],
                   lists: Mapping[str, Iterable[str]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s1 in s1_ids:
            ids = list(dict.fromkeys(i for i in lists.get(s1, ()) if not i.startswith("S1-")))
            f.write(f"{s1}\t{','.join(ids)}\n")
            n += 1
    return n


def write_submission(out_dir: Path, s1_ids: list[str], matches: Mapping[str, Iterable[str]],
                     candidates: Mapping[str, Iterable[str]]) -> None:
    for s1, m in matches.items():
        extra = set(m) - set(candidates.get(s1, ()))
        if extra:
            raise ValueError(f"{s1}: matches not in candidates: {sorted(extra)[:5]}")
    write_id_lists(out_dir / "matching_results.tsv", MATCHING_HEADER, s1_ids, matches)
    write_id_lists(out_dir / "candidate_pairs.tsv", CANDIDATE_HEADER, s1_ids, candidates)


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['cross_encoder_train.py', '--input', '/kaggle/input', '--out-dir', '/kaggle/working', '--pairs', '3000000', '--batch', '64', '--max-train-min', '90']

In [ ]:
"""Cross-encoder experiment, steps 1-3 (GPU): smoke test, 1-epoch fine-tune, borderline validation scoring, blend.

Model: distilbert-base-multilingual-cased (Apache-2.0; the licence is read from the model card and logged).
Training pairs: blocking-v3 train candidates (train_candidates.parquet: the natural mix of true pairs and hard
negatives; no random negatives), a sample of whole S1 lists (~--pairs pairs); 5% of those S1 held out for AUC.
Validation: borderline pairs (0.02 < p_model < 0.98) of the current model's validation probabilities
(val_scores.parquet from run_model_v3.py: models trained on train only). Blend p = (1-w) p_model + w p_ce on
borderline pairs; w and the threshold tuned on the tune half (margin and one owner as the current selection),
reported on the report half. GO only if the report half beats the current model overall AND for US and India,
with precision >= 0.99. Outputs: ce_model/, ce_config.json, ce_val_scores.parquet, ce_report.md.
"""
import argparse
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:
    sys.path.insert(0, "/tmp/src")
from cross_encoder_score import BAND, MAX_LEN, MODEL, blend, find, log, score_pairs, texts  # noqa: E402
from selection import f05_table, select, summary  # noqa: E402

SEED = 42
WS = [0.0, 0.2, 0.35, 0.5, 0.65]
NON_LATIN = r"[\p{L}&&[^\p{Latin}]]"


def auc(y: np.ndarray, p: np.ndarray) -> float | None:
    from sklearn.metrics import roc_auc_score
    return float(roc_auc_score(y, p)) if 0 < y.sum() < len(y) else None


def train(model, tok, pairs: pl.DataFrame, txt: dict, batch: int, lr: float, max_steps: int | None = None) -> float:
    """1 epoch (or max_steps), fp16, AdamW + linear warmup 5%. Returns training pairs per second."""
    import torch
    from transformers import get_linear_schedule_with_warmup
    torch.manual_seed(SEED)
    x = pairs.sample(fraction=1.0, shuffle=True, seed=SEED)
    a, b = [txt.get(s, "") for s in x["s1"].to_list()], [txt.get(c, "") for c in x["cand"].to_list()]
    y = x["is_match"].cast(pl.Float32).to_numpy()
    steps = math.ceil(len(a) / batch) if max_steps is None else min(max_steps, math.ceil(len(a) / batch))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, max(1, int(0.05 * steps)), steps)
    scaler = torch.cuda.amp.GradScaler()
    lossf = torch.nn.BCEWithLogitsLoss()
    model.train()
    t0 = time.time()
    for s in range(steps):
        i = s * batch
        enc = tok(a[i:i + batch], b[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            loss = lossf(model(**enc).logits.float().squeeze(-1), torch.tensor(y[i:i + batch], device="cuda"))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        if s % 500 == 0:
            log(f"  step {s}/{steps} loss {loss.item():.4f}")
    torch.cuda.synchronize()
    return steps * batch / (time.time() - t0)


def main() -> None:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--pairs", type=int, default=400_000)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-5)
    ap.add_argument("--max-train-min", type=float, default=45.0)
    a = ap.parse_args()
    out = a.out_dir
    out.mkdir(parents=True, exist_ok=True)
    rep = ["# Cross-encoder experiment (steps 1-3)\n"]
    log(f"GPU: {torch.cuda.get_device_name(0)}; torch {torch.__version__}")
    try:
        from huggingface_hub import model_info
        lic = getattr(model_info(MODEL).card_data, "license", None)
    except Exception as e:  # the card check must not stop the run; the licence is also recorded in the docs
        lic = f"card check failed: {e!r}"
    log(f"model {MODEL}: licence on the model card = {lic}")
    rep.append(f"Model `{MODEL}`, licence (model card): **{lic}**.\n")

    # ------------------------------------------------------------------ step 1: data + smoke test
    tc = pl.read_parquet(find(a.input, "train_candidates.parquet"), columns=["s1", "cand", "is_match"])
    s1s = tc.select("s1").unique().sort("s1").sample(fraction=1.0, shuffle=True, seed=SEED)
    per = tc.height / s1s.height
    s1s = s1s.head(int(a.pairs / per))
    hold = s1s.with_columns(((pl.col("s1").hash(seed=SEED) % 100) < 5).alias("hold"))
    pairs = tc.join(hold, on="s1")
    txt = texts(a.input, "train", pl.concat([pairs["s1"], pairs["cand"]]).unique())
    tok = AutoTokenizer.from_pretrained(MODEL)
    smp = pairs.sample(2000, seed=SEED)
    lens = [len(x) for x in tok([txt.get(s, "") for s in smp["s1"]], [txt.get(c, "") for c in smp["cand"]], truncation=False)["input_ids"]]
    trunc = float(np.mean(np.array(lens) > MAX_LEN))
    log(f"pairs {pairs.height:,} ({pairs['is_match'].mean():.2f} positive) from {s1s.height:,} S1; held out {int(hold['hold'].sum()):,} S1; "
        f"truncated at {MAX_LEN} tokens: {trunc:.1%} (median length {int(np.median(lens))})")
    m = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to("cuda")
    tr_rate = train(m, tok, smp, txt, a.batch, a.lr)
    t0 = time.time(); score_pairs(m, tok, smp, txt, log_every=0); sc_rate = 2000 / (time.time() - t0)
    n_tr = int((~pairs["hold"]).sum())
    proj = n_tr / tr_rate / 60
    log(f"smoke: train {tr_rate:.0f} pairs/s, score {sc_rate:.0f} pairs/s; full training projected {proj:.1f} min for {n_tr:,} pairs")
    if proj > a.max_train_min:
        keep_s1 = pairs.filter(~pl.col("hold")).select("s1").unique(maintain_order=True).head(int(tr_rate * a.max_train_min * 60 / per))
        pairs = pl.concat([pairs.filter(pl.col("hold")), pairs.filter(~pl.col("hold")).join(keep_s1, on="s1", how="semi")])
        n_tr = int((~pairs["hold"]).sum())
        log(f"cut to {n_tr:,} training pairs (projected {n_tr / tr_rate / 60:.1f} min)")
    rep.append(f"Step 1: {pairs.height:,} pairs; truncated {trunc:.1%}; smoke train {tr_rate:.0f} pairs/s, score {sc_rate:.0f} pairs/s; "
               f"projected training {n_tr / tr_rate / 60:.1f} min.\n")
    del m
    torch.cuda.empty_cache()

    # ------------------------------------------------------------------ step 2: fine-tune (1 epoch)
    m = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to("cuda")
    t0 = time.time()
    rate = train(m, tok, pairs.filter(~pl.col("hold")), txt, a.batch, a.lr)
    ho = pairs.filter(pl.col("hold"))
    p_ho = score_pairs(m, tok, ho, txt, log_every=0)
    ho_auc = auc(ho["is_match"].cast(pl.Int8).to_numpy(), p_ho)
    log(f"fine-tuned in {(time.time() - t0) / 60:.1f} min ({rate:.0f} pairs/s); held-out AUC {ho_auc:.4f} on {ho.height:,} pairs")
    m.save_pretrained(out / "ce_model"); tok.save_pretrained(out / "ce_model")
    rep.append(f"Step 2: trained on {n_tr:,} pairs in {(time.time() - t0) / 60:.1f} min; held-out AUC **{ho_auc:.4f}** ({ho.height:,} pairs).\n")

    # ------------------------------------------------------------------ step 3: borderline validation + blend
    vs = pl.read_parquet(find(a.input, "val_scores.parquet")).select("s1", "cand", "is_match", "p")
    split = pl.read_parquet(find(a.input, "g25_split.parquet"))
    val_s1 = split.filter(pl.col("role") == "val").select(pl.col("source1_entity_id").alias("s1")).join(
        pl.read_parquet(find(a.input, "train_source1.parquet"), columns=["entity_id", "country"]).rename({"entity_id": "s1"}), on="s1", how="left")
    gt = pl.read_parquet(find(a.input, "train_ground_truth.parquet"))
    truth = (gt.join(val_s1, left_on="source1_entity_id", right_on="s1", how="semi")
             .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
             .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != "")))
    half = pl.col("s1").str.extract(r"(\d+)$").cast(pl.Int64) % 2
    s1_tune, s1_rep = val_s1.filter(half == 0), val_s1.filter(half == 1)
    border = vs.filter((pl.col("p") > BAND[0]) & (pl.col("p") < BAND[1]))
    vtxt = texts(a.input, "train", pl.concat([border["s1"], border["cand"]]).unique())
    t0 = time.time()
    p_ce = score_pairs(m, tok, border, vtxt, log_every=0)
    log(f"scored {border.height:,} borderline validation pairs in {time.time() - t0:.0f}s")
    b = border.with_columns(pl.Series("p_ce", p_ce)).join(val_s1, on="s1", how="left")
    names = pl.concat([pl.scan_parquet(find(a.input, f"train_source{k}.parquet")).select("entity_id", "business_name")
                       .filter(pl.col("entity_id").is_in(b["cand"].implode())).collect() for k in (2, 3)])
    s1n = pl.scan_parquet(find(a.input, "train_source1.parquet")).select("entity_id", "business_name").filter(pl.col("entity_id").is_in(b["s1"].implode())).collect()
    from rapidfuzz import fuzz, process
    b = b.join(names.rename({"entity_id": "cand", "business_name": "c_name"}), on="cand", how="left").join(
        s1n.rename({"entity_id": "s1", "business_name": "s_name"}), on="s1", how="left")
    b = b.with_columns(pl.Series("tsr", process.cpdist(b["s_name"].fill_null("").str.to_lowercase().to_list(),
                                                       b["c_name"].fill_null("").str.to_lowercase().to_list(), scorer=fuzz.token_set_ratio, workers=-1)))
    rows = []
    for gname, cond in {"all borderline": pl.lit(True), "US": pl.col("country") == "US", "India": pl.col("country") == "India",
                        "Indian-script candidate": pl.col("c_name").str.contains(NON_LATIN), "low name similarity (tsr<70)": pl.col("tsr") < 70}.items():
        d = b.filter(cond)
        y = d["is_match"].cast(pl.Int8).to_numpy()
        rows.append({"group": gname, "pairs": d.height, "true": int(y.sum()), "auc_ce": auc(y, d["p_ce"].to_numpy()), "auc_model": auc(y, d["p"].to_numpy()),
                     "auc_blend_0.5": auc(y, 0.5 * d["p"].to_numpy() + 0.5 * d["p_ce"].to_numpy())})
    auc_tab = pl.DataFrame(rows)
    log("borderline AUC:\n" + str(auc_tab))

    cur = {"margin": 0.7, "one_owner": True}  # current selection (model v3): tau re-tuned, margin and owner kept
    full = vs.join(b.select("s1", "cand", "p_ce"), on=["s1", "cand"], how="left")
    pm, pc = full["p"].to_numpy(), full["p_ce"].fill_null(np.nan).to_numpy()
    grid = []
    for w in WS:
        sv = full.select("s1", "cand").with_columns(pl.Series("p", blend(pm, pc, w)))
        for tau in [round(x, 3) for x in np.arange(0.30, 0.91, 0.025)]:
            f = f05_table(select(sv, tau=tau, **cur), truth, s1_tune)["f05"].mean()
            grid.append({"w": w, "tau": tau, "tune_f05": f})
    g = pl.DataFrame(grid).sort("tune_f05", descending=True)
    res = []
    for w in WS:
        bw = g.filter(pl.col("w") == w).row(0, named=True)
        sv = full.select("s1", "cand").with_columns(pl.Series("p", blend(pm, pc, w)))
        sm = summary(f05_table(select(sv, tau=bw["tau"], **cur), truth, s1_rep), by="country")
        gg = {r["group"]: r for r in sm.iter_rows(named=True)}
        res.append({"w": w, "tau": bw["tau"], "tune_f05": bw["tune_f05"], **{k: gg[k]["macro_f05"] for k in gg},
                    "precision": gg["overall"]["precision"], "recall": gg["overall"]["recall"]})
    res = pl.DataFrame(res)
    base = res.filter(pl.col("w") == 0.0).row(0, named=True)
    best = res.filter(pl.col("w") > 0).sort("tune_f05", descending=True).row(0, named=True)
    go = (best["overall"] > base["overall"] and best["country=US"] > base["country=US"] and best["country=India"] > base["country=India"]
          and best["precision"] >= 0.99)
    log(f"blend results:\n{res}\nGO = {go} (best w {best['w']}: report {best['overall']:.4f} vs current {base['overall']:.4f})")
    full.write_parquet(out / "ce_val_scores.parquet")
    (out / "ce_config.json").write_text(json.dumps({"model": MODEL, "licence": str(lic), "max_len": MAX_LEN, "band": BAND,
                                                    "w": best["w"], "selection": {"tau": best["tau"], **cur}, "go": go,
                                                    "heldout_auc": ho_auc, "train_pairs": n_tr, "truncated_share": trunc}, indent=1, default=str))
    rep += ["Step 3: borderline validation pairs (0.02 < p < 0.98): " + f"{border.height:,}\n", str(auc_tab) + "\n",
            "Blend (tau tuned on the tune half; report half):\n", str(res) + "\n",
            f"**GO = {go}** (rule: beat w = 0 overall, US and India on the report half, precision >= 0.99).\n"]
    (out / "ce_report.md").write_text("\n".join(rep))
    log("done")


if __name__ == "__main__":
    main()
